# Model XGBoost – prognoza popytu dla restauracji (HoReCa)

Notebook trenuje model XGBoost na zbiorze `Dane_restauracje.csv` (wynik notebooka 04), ocenia go na danych z 2025 roku, zapisuje prognozy i porównuje je z prognozami Random Forest (notebook 07).

**Załadowanie bibliotek oraz wczytanie wcześniej przygotowanego zbioru danych**

In [1]:
import warnings
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import holidays

import xgboost as xgb

from sklearn.metrics import (mean_absolute_error,mean_squared_error,r2_score)
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')

# Wczytanie danych
df = pd.read_csv(r'../dane/przetworzone/Dane_restauracje.csv', sep=';',decimal=',',encoding='utf-8-sig')

**Kodowanie wartości tekstowych na liczbowe**

Zmienne tekstowe, takie jak klient oraz podkategoria produktu, zostały zamienione na wartości liczbowe. Jest to wymagane, ponieważ wykorzystywany algorytm XGBoost pracuje wyłącznie na danych numerycznych.
Dla każdej zmiennej kategorii przypisano unikalny kod liczbowy za pomocą `cat.codes`. Słownik `mapowania` zapamiętuje przypisanie kodów, aby na końcu przywrócić oryginalne nazwy klientów i podkategorii.

In [2]:
mapowania = {}
kolumny_tekstowe = ['Klient', 'Podkategoria']

for col in kolumny_tekstowe:
    cat = df[col].astype('category')
    mapowania[col] = dict(enumerate(cat.cat.categories))
    df[col] = cat.cat.codes

print(df[['Klient', 'Podkategoria']].head())

   Klient  Podkategoria
0       0             0
1       1             0
2       2             0
3       3             0
4       4             0


**Sortowanie, ustawienie indexu (data), podział zbioru danych**

Dane zostały posortowane względem daty oraz podkategorii produktu. Kolejno ustawiono `data_sprzedaży` jako indeks, aby umożliwić chronologiczny podział danych (bez losowego mieszania – zapobiega to wyciekowi informacji z przyszłości).

**Podział danych na zbiory**
* Zbiór treningowy (2021–2023) – służy do nauki zależności na danych historycznych,
* Zbiór walidacyjny (2024) – służy do monitorowania jakości modelu i doboru liczby drzew (early stopping),
* Zbiór testowy (2025) – zbiór końcowy do oceny skuteczności modelu na danych, których model nie widział.

In [3]:
# Sortowanie i ustawienie indexu
df = df.sort_values(["data_sprzedaży","Podkategoria"])
df = df.set_index('data_sprzedaży')

df_model = df

# Podział zbioru danych
train = df_model.loc['2021-01-01':'2023-12-31']   # rok treningu
val   = df_model.loc['2024-01-01':'2024-12-31']   # rok walidacji
test  = df_model.loc['2025-01-01':'2025-12-31']   # rok testu 

print(f"Rozmiar zbiorów:")
print(f"Train: {train.shape[0]:,} wierszy")
print(f"Val: {val.shape[0]:,} wierszy")
print(f"Test: {test.shape[0]:,} wierszy")

Rozmiar zbiorów:
Train: 279,225 wierszy
Val: 93,330 wierszy
Test: 93,075 wierszy


**Grupowanie danych na zmienne cechy i cel**

Zmienną celu, czyli prognozowaną wartością, jest `Ilość` (dzienna sprzedaż w kg).
Pozostałe zmienne z listy `FEATURES` (23 cech: klient, podkategoria, cechy cykliczne i kalendarzowe, flagi sezonowe oraz `lag_364`) służą jako zmienne opisujące zjawisko sprzedaży. Oprócz cech wspólnych z modelem dla sklepów wykorzystano cechy przygotowane specjalnie dla restauracji: `czy_okazja`, `czy_przed_okazja`, `czy_szczyt_restauracja` i `czy_czwartek`.

Na tej podstawie przygotowano macierze cech `X` i wektory celu `y` dla zbioru treningowego, walidacyjnego i testowego.

In [4]:
FEATURES = ['Klient', 'Podkategoria', 
            # Cykliczność
            'miesiac_sin', 'miesiac_cos',
            'kwartał_sin', 'kwartał_cos',
            'day_sin', 'day_cos', 
            # Cechy kalendarza
            'tydzien_roku',
            'rok', 'czy_niedziela', 'okres_wielkanocny',
            'okres_wigilijny','czy_ferie_slask', 
            'czy_wakacje', 'czy_majowka', 'czy_swieto',
            'dni_do_swieta', 'czy_okazja', 'czy_przed_okazja',
            'czy_szczyt_restauracja', 'czy_czwartek', 
            # Lag
            'lag_364']

X_train = train[FEATURES]
y_train = train['Ilość']
y_val = val['Ilość']

X_val = val[FEATURES]

X_test = test[FEATURES]
y_test = test['Ilość']

**Definiowanie funkcji do oceny jakości modelu**

Funkcja `print_metrics` wylicza:
* **MAE** – średni błąd bezwzględny [kg],
* **RMSE** – pierwiastek błędu średniokwadratowego [kg], silniej karze duże pomyłki,
* **R²** – jaka część zmienności sprzedaży jest wyjaśniona przez model,
* **Bias** – średnia różnica *prognoza − rzeczywistość*; wartość ujemna oznacza niedoszacowanie popytu,
* **MAE / średnia** – błąd w odniesieniu do średniej sprzedaży,
* **WMAPE** – ważony procentowy błąd prognozy, liczony tylko dla dni ze sprzedażą większą od zera (dni zerowe zaburzają tę statystykę).

In [5]:
# Definiowanie parametrów badanych do oceny modelu 
def wmape(y_fact, y_pred):
    # Wykluczenie wartości 0 ponieważ zaburza statystykę 
    mask = y_fact > 0
    return np.sum(np.abs(y_fact[mask] - y_pred[mask])) / np.sum(y_fact[mask]) * 100

def print_metrics(name, y_fact, y_pred):

    mae  = mean_absolute_error(y_fact, y_pred)
    rmse = np.sqrt(mean_squared_error(y_fact, y_pred))
    r2   = r2_score(y_fact, y_pred)

    bias = np.mean(y_pred - y_fact)

    mean_ = y_fact.mean()
    wm    = wmape(y_fact.values, y_pred)

    print(f"\n{'─'*20}")
    print(f"  {name}")
    print(f"{'─'*20}")
    print(f"MAE           : {mae:.4f} kg")
    print(f"RMSE          : {rmse:.4f} kg")
    print(f"R²            : {r2:.4f}")
    print(f"Bias          : {bias:.4f} kg")
    print(f"Średnia       : {mean_:.4f} kg")
    print(f"MAE / średnia : {mae/mean_*100:.1f}%")
    print(f"WMAPE         : {wm:.1f}%")

    return mae, rmse, r2, bias

**Budowa modelu XGBoost**

Do stworzenia prognozy wykorzystano model XGBRegressor z funkcją celu `reg:tweedie`, która dobrze radzi sobie z danymi sprzedażowymi, gdzie występuje dużo wartości zerowych oraz dodatnie wartości ciągłe. Zastosowano te same hiperparametry co w modelu dla sklepów (`max_depth=4`, `learning_rate=0.02`, regularyzacja L1/L2). Parametr `early_stopping_rounds` pozwala dobrać odpowiednią liczbę drzew – trening na latach 2021–2023 zatrzymuje się, gdy przez 50 iteracji wynik na zbiorze walidacyjnym 2024 się nie poprawia.

In [6]:
params = dict(
    n_estimators = 2000,      # górny limit, early stopping zatrzyma wcześniej
    max_depth = 4,
    learning_rate = 0.02,
    objective = 'reg:tweedie',
    tweedie_variance_power = 1.6,
    tree_method = 'hist',
    random_state = 42,
    reg_alpha = 1,
    reg_lambda = 10,
    early_stopping_rounds = 50,
)

In [ ]:
print("Trening 2021–2023, Early stopping na zbiorze walidacyjnym 2024")

reg = xgb.XGBRegressor(**params)

reg.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],   # TYLKO val, nie train, żeby nie zawyżać metryk
    verbose=False
)

najlepsze_liczba_drzew = reg.best_iteration + 1
print(f"Optymalna liczba drzew : {najlepsze_liczba_drzew}")

# predykcja zbioru walidacyjnego
pred_val = reg.predict(X_val).clip(0)
# Metryki 
mae_val, rmse_val, r2_val, bias_val = print_metrics("WALIDACJA 2024", y_val, pred_val)

Trening 2021–2023, Early stopping na zbiorze walidacyjnym 2024
Optymalna liczba drzew : 330

────────────────────
  WALIDACJA 2024
────────────────────
MAE           : 1.6586 kg
RMSE          : 16.6759 kg
R²            : 0.4903
Bias          : -0.6250 kg
Średnia       : 1.9491 kg
MAE / średnia : 85.1%
WMAPE         : 68.2%


**Model końcowy i test na danych z 2025 roku**

Zbiór treningowy i walidacyjny (2021–2024) zostały połączone, a model końcowy wytrenowano z liczbą drzew wyznaczoną wcześniej przez early stopping (parametr `early_stopping_rounds` jest usuwany, bo model końcowy nie ma już zbioru walidacyjnego). Ujemne prognozy obcinane są do 0, a wartości poniżej 0,01 kg zaokrąglane do 0. Skuteczność oceniono na zbiorze testowym 2025.

In [8]:
# Tworzenie modelu końcowego TRAIN+VAL, TEST NA 2025

print("Trening na 2021–2024, Test końcowy na zbiorze danych 2025")

# Połączenie zbioru treningowego i validacyjnego
X_train_full = pd.concat([X_train, X_val])
y_train_full = pd.concat([y_train, y_val])

# Użycie najlepsze_liczba_drzew z modelu 1
params_final = params.copy()
params_final["n_estimators"] = najlepsze_liczba_drzew

# Usunięcie z parametru early_stopping_rounds
params_final.pop("early_stopping_rounds")
 
# Tworzenie modelu końcowego
reg_final = xgb.XGBRegressor(**params_final)

# Trenowanie
reg_final.fit(X_train_full, y_train_full, verbose=False)

# Predykcja na zbiorze testowym
pred_test = reg_final.predict(X_test).clip(0)
pred_test = np.clip(reg_final.predict(X_test), 0, None)
pred_test = np.where(pred_test < 0.01, 0, pred_test)

# Ocena zbioru testowego 
mae_test, rmse_test, r2_test, bias_test = print_metrics("TEST 2025", y_test, pred_test)

# Zapisanie prognozy 
test = test.copy()
test["Prognoza"] = pred_test
test["Rzeczywista"] = y_test.values

Trening na 2021–2024, Test końcowy na zbiorze danych 2025

────────────────────
  TEST 2025
────────────────────
MAE           : 1.5184 kg
RMSE          : 13.1218 kg
R²            : 0.5706
Bias          : -0.2782 kg
Średnia       : 1.7761 kg
MAE / średnia : 85.5%
WMAPE         : 64.7%


**Analiza ważności cech**

Wbudowana ważność cech XGBoost dla modelu końcowego: 10 najważniejszych zmiennych. Dominującą cechą jest `lag_364`, a kolejne miejsca zajmują flaga `czy_czwartek`, podkategoria i `czy_niedziela`.

In [13]:
xgb_importance = pd.DataFrame({
    'Cecha': X_train_full.columns,
    'Ważność': reg_final.feature_importances_
})

xgb_importance = xgb_importance.sort_values(
    by='Ważność',
    ascending=False
)
print('Analiza ważności cech dla XGBoost / Restauracji')
print(xgb_importance.head(10))

Analiza ważności cech dla XGBoost / Restauracji
            Cecha   Ważność
22        lag_364  0.521462
21   czy_czwartek  0.091029
1    Podkategoria  0.086920
10  czy_niedziela  0.082592
0          Klient  0.053412
6         day_sin  0.050639
7         day_cos  0.022478
16     czy_swieto  0.021907
9             rok  0.019306
4     kwartał_sin  0.014864


**Obliczenie błędów prognozy i odkodowanie zmiennych tekstowych**

Dla każdego wiersza liczony jest błąd (prognoza − rzeczywistość) oraz błąd bezwzględny, dodawane są etykiety modelu i segmentu, a kody klientów i podkategorii zamieniane są z powrotem na nazwy. Wynik zapisywany jest do `../wyniki/prognozy/XGB_Restauracje_Prognoza_2025.csv` w celu dalszej analizy w Power BI.

In [ ]:
test["Błąd"] = test["Prognoza"] - test["Rzeczywista"]
test["Błąd_abs"] = abs(test["Błąd"])


# Przywrócenie nazw sklepów i podkategorii
test["Klient_nazwa"] = test["Klient"].map(mapowania["Klient"])
test["Podkategoria_nazwa"] = test["Podkategoria"].map(mapowania["Podkategoria"])
test.reset_index(inplace=True)

In [54]:
test["Model"] = "XGBoost"
test["Segment"] = "Restauracje"

In [55]:
xgb_rest = test[['data_sprzedaży','Prognoza', 'Model',
       'Segment', 'Rzeczywista', 'Błąd', 'Błąd_abs', 'Klient_nazwa','Podkategoria_nazwa']]

In [56]:
xgb_rest.to_csv(
    r"../wyniki/prognozy/XGB_Restauracje_Prognoza_2025.csv",
    sep=";",
    decimal=",",
    index=False,
    encoding="utf-8-sig"
)

**Porównanie prognoz Random Forest i XGBoost**

Wczytywane są zapisane prognozy obu modeli na 2025 rok (plik `RF_Restauracje_Prognoza_2025.csv` pochodzi z notebooka 07, więc musi on zostać uruchomiony wcześniej). Kolumny prognoz i błędów otrzymują sufiksy `_RF_R` / `_XGB_R`, a tabele łączone są po dacie, segmencie, kliencie, podkategorii i wartości rzeczywistej. Dodatkowo liczona jest różnica prognoz RF − XGB. Wynik zapisywany jest do `../wyniki/prognozy/Porownanie_RF_Restauracji_XGB_2025.csv` (do analizy w Power BI).

In [ ]:
import pandas as pd

xgb = pd.read_csv(
    r"../wyniki/prognozy/XGB_Restauracje_Prognoza_2025.csv",
    sep=";",
    decimal=",",
    index=False,
    encoding="utf-8-sig"
)

rf = pd.read_csv(
    r"../wyniki/prognozy/RF_Restauracje_Prognoza_2025.csv",
    sep=";",
    decimal=",",
    index=False,
    encoding="utf-8-sig"
)

In [60]:
rf = rf.rename(columns={
    "Prognoza": "Prognoza_RF_R",
    "Błąd": "Błąd_RF_R",
    "Błąd_abs": "Błąd_abs_RF_R"
})

xgb = xgb.rename(columns={
    "Prognoza": "Prognoza_XGB_R",
    "Błąd": "Błąd_XGB_R",
    "Błąd_abs": "Błąd_abs_XGB_R"
})

In [61]:
porownanie = rf.merge(
    xgb[[
        "data_sprzedaży",
        "Segment",
        "Rzeczywista",
        "Klient_nazwa",
        "Podkategoria_nazwa",
        "Prognoza_XGB_R",
        "Błąd_XGB_R",
        "Błąd_abs_XGB_R"
    ]],
    on=[
        "data_sprzedaży",
        "Segment",
        "Rzeczywista",
        "Klient_nazwa",
        "Podkategoria_nazwa"
    ],
    how="inner"
)
for col in ["Prognoza_RF_R", "Prognoza_XGB_R"]:
    porownanie[col] = (
        porownanie[col]
        .astype(str)
        .str.replace(",", ".", regex=False)
    )
    porownanie[col] = pd.to_numeric(porownanie[col], errors="coerce")

porownanie["Różnica_prognoz_RF_minus_XGB"] = (
    porownanie["Prognoza_RF_R"] - porownanie["Prognoza_XGB_R"]
)

In [ ]:
porownanie.to_csv(
    r"../wyniki/prognozy/Porownanie_RF_Restauracji_XGB_2025.csv",
    sep=";",
    decimal=",",
    index=False,
    encoding="utf-8-sig"
)